## 2. Detection & Segmentation

In [ ]:
!pip install pycocotools opencv-python matplotlib timm ultralytics segmentation-models-pytorch torchmetrics -q

In [ ]:
import os
import json
from tqdm import tqdm

def extract_categories(anno_dir):

    categories = set()

    files = os.listdir(anno_dir)

    for file in tqdm(files):

        if not file.endswith(".json"):
            continue

        path = os.path.join(anno_dir, file)

        with open(path) as f:
            data = json.load(f)

        for key in data:

            if "item" not in key:
                continue

            item = data[key]

            if "category_name" in item:
                categories.add(item["category_name"])

    return sorted(list(categories))

In [ ]:
os.chdir('/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd')

In [ ]:
import os
import json
from collections import Counter
from tqdm import tqdm

train_anno_dir = "./train/train/train/annos"

category_counter = Counter()

files = os.listdir(train_anno_dir)

for file in tqdm(files):

    if not file.endswith(".json"):
        continue

    path = os.path.join(train_anno_dir, file)

    with open(path) as f:
        data = json.load(f)

    for key in data:

        if "item" not in key:
            continue

        category = data[key]["category_name"]

        category_counter[category] += 1

In [ ]:
print("Category frequencies:\n")

for cat, count in category_counter.most_common():
    print(cat, ":", count)

In [ ]:
TOP5 = [cat for cat, _ in category_counter.most_common(5)]

print("Top 5 categories:")
print(TOP5)

In [ ]:
label_map = {cat:i for i,cat in enumerate(TOP5)}

print("Label mapping:")

for k,v in label_map.items():
    print(k,"->",v)

In [ ]:
import os, json, cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE   = 512
NUM_CLASSES = 5   # reuses label_map and TOP5 from Section 1

TRAIN_IMG  = "/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd/train/train/train/image"
TRAIN_ANNO = "/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd/train/train/train/annos"
VAL_IMG    = "/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd/validation/validation/validation/image"
VAL_ANNO   = "/kaggle/input/datasets/mightyshashank/vr-project-1-dataset-non-pwd/validation/validation/validation/annos"

train_df = pd.read_csv("/kaggle/input/datasets/mightyshashank/vr-csv/train_labels_top5_new.csv")
val_df   = pd.read_csv("/kaggle/input/datasets/mightyshashank/vr-csv/val_labels_top5_new.csv")


In [ ]:
def parse_anno(img_name, anno_dir, label_map):
    path = os.path.join(anno_dir, img_name.replace(".jpg", ".json"))
    items = []
    if not os.path.exists(path):
        return items
    with open(path) as f:
        data = json.load(f)
    for key in data:
        if "item" not in key:
            continue
        item = data[key]
        cat  = item.get("category_name", "")
        if cat not in label_map:
            continue
        bbox = item.get("bounding_box", None)
        segs = item.get("segmentation", [])
        if bbox is None or not segs:
            continue
        items.append({"label": label_map[cat], "bbox": bbox, "segmentation": segs})
    return items

def poly_to_mask(segmentation, H, W):
    mask = np.zeros((H, W), dtype=np.uint8)
    for poly in segmentation:
        pts = np.array(poly, dtype=np.float32).reshape(-1, 2).astype(np.int32)
        cv2.fillPoly(mask, [pts], 1)
    return mask

def collate_fn(batch):
    return tuple(zip(*batch))


### 2.1 Mask R-CNN (Transfer Learning)

In [ ]:
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn  import MaskRCNNPredictor

class MRCNNDataset(Dataset):
    def __init__(self, df, img_dir, anno_dir, size=IMG_SIZE):
        self.df = df; self.img_dir = img_dir
        self.anno_dir = anno_dir; self.size = size

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image"]
        img = Image.open(os.path.join(self.img_dir, img_name)).convert("RGB")
        W, H = img.size
        img  = img.resize((self.size, self.size))
        img_t = TF.to_tensor(img)
        sx, sy = self.size/W, self.size/H

        boxes, labels, masks = [], [], []
        for ann in parse_anno(img_name, self.anno_dir, label_map):
            x1,y1,x2,y2 = [c*s for c,s in zip(ann["bbox"],[sx,sy,sx,sy])]
            if x2 <= x1 or y2 <= y1: continue
            sc = []
            for poly in ann["segmentation"]:
                pts = np.array(poly).reshape(-1,2)
                pts[:,0] *= sx; pts[:,1] *= sy
                sc.append(pts.flatten().tolist())
            boxes.append([x1,y1,x2,y2])
            labels.append(ann["label"] + 1)          # 0 = background
            masks.append(poly_to_mask(sc, self.size, self.size))

        if len(boxes) == 0:
            t = {"boxes":  torch.zeros((0,4), dtype=torch.float32),
                 "labels": torch.zeros((0,),  dtype=torch.int64),
                 "masks":  torch.zeros((0,self.size,self.size), dtype=torch.uint8)}
        else:
            t = {"boxes":  torch.tensor(boxes,  dtype=torch.float32),
                 "labels": torch.tensor(labels, dtype=torch.int64),
                 "masks":  torch.tensor(np.stack(masks), dtype=torch.uint8)}
        t["image_id"] = torch.tensor([idx])
        return img_t, t

train_mrcnn_dl = DataLoader(MRCNNDataset(train_df, TRAIN_IMG, TRAIN_ANNO),
                             batch_size=4, shuffle=True,  collate_fn=collate_fn, num_workers=2)
val_mrcnn_dl   = DataLoader(MRCNNDataset(val_df,   VAL_IMG,   VAL_ANNO),
                             batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)


In [ ]:
def get_maskrcnn():
    model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
    in_f = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor  = FastRCNNPredictor(in_f, NUM_CLASSES + 1)
    in_m = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_m, 256, NUM_CLASSES + 1)
    return model

maskrcnn   = get_maskrcnn().to(DEVICE)
opt_mrcnn  = torch.optim.Adam([p for p in maskrcnn.parameters() if p.requires_grad], lr=1e-4)


In [ ]:
EPOCHS = 5
best_mrcnn_loss = float("inf")

for epoch in range(EPOCHS):
    maskrcnn.train()
    total = 0
    for imgs, targets in train_mrcnn_dl:
        imgs    = [i.to(DEVICE) for i in imgs]
        targets = [{k: v.to(DEVICE) for k,v in t.items()} for t in targets]
        loss    = sum(maskrcnn(imgs, targets).values())
        opt_mrcnn.zero_grad(); loss.backward(); opt_mrcnn.step()
        total += loss.item()
    avg = total / len(train_mrcnn_dl)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg:.4f}")
    if avg < best_mrcnn_loss:
        best_mrcnn_loss = avg
        torch.save(maskrcnn.state_dict(), "/kaggle/working/best_maskrcnn.pth")


In [ ]:
from torchmetrics.detection import MeanAveragePrecision

maskrcnn.load_state_dict(torch.load("/kaggle/working/best_maskrcnn.pth"))
maskrcnn.eval()

metric_mrcnn = MeanAveragePrecision(iou_type="segm")
iou_cls_mrcnn = [[] for _ in range(NUM_CLASSES)]

with torch.no_grad():
    for imgs, targets in val_mrcnn_dl:
        imgs       = [i.to(DEVICE) for i in imgs]
        preds      = maskrcnn(imgs)
        metric_mrcnn.update([{k: v.cpu() for k,v in p.items()} for p in preds],
                             [{k: v.cpu() for k,v in t.items()} for t in targets])
        for pred, tgt in zip(preds, targets):
            for pm, pl in zip(pred["masks"], pred["labels"]):
                pb = (pm.squeeze(0).cpu().numpy() > 0.5).astype(np.uint8)
                c  = pl.item() - 1
                if not (0 <= c < NUM_CLASSES): continue
                gt = tgt["masks"][tgt["labels"] == (c+1)].cpu().numpy()
                if len(gt) == 0: continue
                iou_cls_mrcnn[c].append(max((pb & g).sum()/((pb | g).sum()+1e-6) for g in gt))

res_mrcnn   = metric_mrcnn.compute()
per_iou_mrcnn  = [np.mean(v) if v else 0.0 for v in iou_cls_mrcnn]
miou_mrcnn  = np.mean(per_iou_mrcnn)
dice_mrcnn  = np.mean([2*iou/(1+iou) for iou in per_iou_mrcnn])

print("Mask R-CNN Results")
print(f"mAP@[0.5:0.95] (segm) : {res_mrcnn['map'].item():.4f}")
print(f"mAP@0.50       (segm) : {res_mrcnn['map_50'].item():.4f}")
print(f"mIoU                  : {miou_mrcnn:.4f}")
print(f"Dice (macro)          : {dice_mrcnn:.4f}")
print(f"Per-class IoU         : {[f'{v:.3f}' for v in per_iou_mrcnn]}")


### 2.2 YOLO (Transfer Learning — YOLOv8s-seg)

In [ ]:
import yaml
from ultralytics import YOLO

YOLO_DIR = "/kaggle/working/yolo_data"

def build_yolo_split(df, img_dir, anno_dir, split):
    os.makedirs(f"{YOLO_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{YOLO_DIR}/labels/{split}", exist_ok=True)
    for _, row in df.iterrows():
        name = row["image"]
        src  = os.path.join(img_dir, name)
        if not os.path.exists(src): continue
        img  = cv2.imread(src)
        H, W = img.shape[:2]
        stem = os.path.splitext(name)[0]
        cv2.imwrite(f"{YOLO_DIR}/images/{split}/{stem}.jpg", img)
        lines = []
        for ann in parse_anno(name, anno_dir, label_map):
            cls, pts_flat = ann["label"], []
            for poly in ann["segmentation"]:
                arr = np.array(poly).reshape(-1,2).astype(float)
                arr[:,0] /= W; arr[:,1] /= H
                pts_flat.extend(arr.flatten().tolist())
            if pts_flat:
                lines.append(f"{cls} " + " ".join(f"{v:.6f}" for v in pts_flat))
        with open(f"{YOLO_DIR}/labels/{split}/{stem}.txt","w") as f:
            f.write("\n".join(lines))

build_yolo_split(train_df, TRAIN_IMG, TRAIN_ANNO, "train")
build_yolo_split(val_df,   VAL_IMG,   VAL_ANNO,   "val")

yaml.dump({"path": YOLO_DIR, "train": "images/train", "val": "images/val",
           "nc": NUM_CLASSES, "names": list(label_map.keys())},
          open(f"{YOLO_DIR}/data.yaml","w"))
print("YOLO dataset ready")


In [ ]:
yolo_model = YOLO("yolov8s-seg.pt")
yolo_model.train(
    data   = f"{YOLO_DIR}/data.yaml",
    epochs = 10,
    imgsz  = IMG_SIZE,
    batch  = 8,
    device = 0 if torch.cuda.is_available() else "cpu",
    project = "/kaggle/working",
    name    = "yolo_seg",
    exist_ok = True
)


In [ ]:
val_res_yolo = yolo_model.val(data=f"{YOLO_DIR}/data.yaml")

map_box_yolo = val_res_yolo.box.map
map_seg_yolo = val_res_yolo.seg.map

print("YOLO Results")
print(f"Box  mAP@[0.5:0.95] : {map_box_yolo:.4f}")
print(f"Seg  mAP@[0.5:0.95] : {map_seg_yolo:.4f}")
print(f"Box  mAP@0.50       : {val_res_yolo.box.map50:.4f}")
print(f"Seg  mAP@0.50       : {val_res_yolo.seg.map50:.4f}")


### 2.3 U-Net (Transfer Learning — ResNet34 encoder)

In [ ]:
import segmentation_models_pytorch as smp

class UNetDataset(Dataset):
    def __init__(self, df, img_dir, anno_dir, size=IMG_SIZE):
        self.df = df; self.img_dir = img_dir
        self.anno_dir = anno_dir; self.size = size

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image"]
        img = Image.open(os.path.join(self.img_dir, img_name)).convert("RGB")
        W, H = img.size
        img  = img.resize((self.size, self.size))
        img_t = TF.to_tensor(img)
        sx, sy = self.size/W, self.size/H

        seg_mask = np.zeros((self.size, self.size), dtype=np.int64)
        for ann in parse_anno(img_name, self.anno_dir, label_map):
            sc = []
            for poly in ann["segmentation"]:
                pts = np.array(poly).reshape(-1,2)
                pts[:,0] *= sx; pts[:,1] *= sy
                sc.append(pts.flatten().tolist())
            m = poly_to_mask(sc, self.size, self.size)
            seg_mask[m == 1] = ann["label"] + 1   # 0 = background
        return img_t, torch.tensor(seg_mask, dtype=torch.long)

train_unet_dl = DataLoader(UNetDataset(train_df, TRAIN_IMG, TRAIN_ANNO),
                            batch_size=8, shuffle=True,  num_workers=2)
val_unet_dl   = DataLoader(UNetDataset(val_df,   VAL_IMG,   VAL_ANNO),
                            batch_size=8, shuffle=False, num_workers=2)


In [ ]:
unet     = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet",
                    in_channels=3, classes=NUM_CLASSES+1).to(DEVICE)
ce_loss  = torch.nn.CrossEntropyLoss()
opt_unet = torch.optim.Adam(unet.parameters(), lr=1e-4)

EPOCHS = 5
best_unet_loss = float("inf")

for epoch in range(EPOCHS):
    unet.train()
    total = 0
    for imgs, masks in train_unet_dl:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        loss = ce_loss(unet(imgs), masks)
        opt_unet.zero_grad(); loss.backward(); opt_unet.step()
        total += loss.item()
    avg = total / len(train_unet_dl)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg:.4f}")
    if avg < best_unet_loss:
        best_unet_loss = avg
        torch.save(unet.state_dict(), "/kaggle/working/best_unet.pth")


In [ ]:
from scipy import ndimage

unet.load_state_dict(torch.load("/kaggle/working/best_unet.pth"))
unet.eval()

iou_cls_unet  = [[] for _ in range(NUM_CLASSES)]
dice_cls_unet = [[] for _ in range(NUM_CLASSES)]

with torch.no_grad():
    for imgs, masks in val_unet_dl:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds = unet(imgs).argmax(dim=1)
        for c in range(NUM_CLASSES):
            p = (preds == c+1).cpu().numpy()
            g = (masks == c+1).cpu().numpy()
            inter = (p & g).sum(); union = (p | g).sum()
            if union > 0:
                iou_cls_unet[c].append(inter / (union + 1e-6))
                dice_cls_unet[c].append(2*inter / (p.sum() + g.sum() + 1e-6))

per_iou_unet  = [np.mean(v) if v else 0.0 for v in iou_cls_unet]
per_dice_unet = [np.mean(v) if v else 0.0 for v in dice_cls_unet]
miou_unet = np.mean(per_iou_unet)
dice_unet = np.mean(per_dice_unet)

print("U-Net Results")
print(f"mIoU         : {miou_unet:.4f}")
print(f"Dice (macro) : {dice_unet:.4f}")
print(f"Per-class IoU: {[f'{v:.3f}' for v in per_iou_unet]}")

# Instance post-processing via connected components
def unet_instances(pred_mask):
    instances = []
    for c in range(NUM_CLASSES):
        binary  = (pred_mask == c+1).astype(np.uint8)
        labeled, n = ndimage.label(binary)
        for i in range(1, n+1):
            comp = (labeled == i)
            ys, xs = np.where(comp)
            if len(xs) < 10: continue
            instances.append({
                "bbox":  [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())],
                "mask":  comp.astype(np.uint8), "label": c})
    return instances


### 2.4 Summary Table

In [ ]:
summary = pd.DataFrame({
    "Model"            : ["Mask R-CNN", "YOLOv8s-seg", "U-Net"],
    "mAP@[0.5:0.95]"   : [f"{res_mrcnn['map'].item():.4f}", f"{map_seg_yolo:.4f}", "N/A"],
    "mIoU"             : [f"{miou_mrcnn:.4f}",  "—",  f"{miou_unet:.4f}"],
    "Dice (macro)"     : [f"{dice_mrcnn:.4f}",  "—",  f"{dice_unet:.4f}"],
})
print(summary.to_string(index=False))
